# dots.mocr — Kaggle GPU Server

> Source: [github.com/rednote-hilab/dots.mocr](https://github.com/rednote-hilab/dots.mocr)

**Trước khi chạy:** Vào Settings (góc phải) → Accelerator → **GPU T4 x1**

---
**Thứ tự chạy mỗi session:**  Cell 1 → Cell 2 → Cell 3 → Cell 3b → Cell 4a → Cell 4b → Cell 5

## Cell 1 — Kiểm tra GPU + cài packages

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

!pip install "vllm>=0.11.0,<0.15.0" pyngrok xformers -q
import vllm, torch
print(f"✅ vLLM {vllm.__version__} | PyTorch {torch.__version__} | CUDA {torch.version.cuda}")

cuda_ver = torch.version.cuda.replace(".", "")
torch_ver = f"{torch.__version__.split('.')[0]}.{torch.__version__.split('.')[1]}"
wheel_url = f"https://flashinfer.ai/whl/cu{cuda_ver}/torch{torch_ver}/"
r = subprocess.run(["pip", "install", "flashinfer-python", "-i", wheel_url, "-q"],
                   capture_output=True, text=True)
print("✅ FlashInfer done" if r.returncode == 0 else f"⚠️ FlashInfer skip: {r.stderr[-80:]}")

## Cell 2 — Clone repo + cài package + tải model (~4GB, 5–10 phút)

In [ ]:
import os, shutil

# ── Tuỳ chọn model ──────────────────────────────────────────────────────────
USE_V15          = False  # True  → dùng kristaller486/dots.ocr-1.5
                          # False → dùng rednote-hilab/dots.mocr  (chính thức)
FORCE_REDOWNLOAD = False  # True  → xóa model cũ, tải lại (dùng khi đổi model)
# ────────────────────────────────────────────────────────────────────────────

WORK_DIR  = "/kaggle/working/dots.mocr"
MODEL_DIR = f"{WORK_DIR}/weights/DotsMOCR"

# Clone repo chính thức
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/rednote-hilab/dots.mocr.git {WORK_DIR}

%cd {WORK_DIR}
!git pull

# Cài package dots_ocr + dependencies
!pip install -e . -q

# Xóa model cũ nếu muốn đổi
if FORCE_REDOWNLOAD and os.path.exists(MODEL_DIR):
    shutil.rmtree(MODEL_DIR)
    print("🗑️ Đã xóa model cũ")

# Tải model
if not os.path.exists(MODEL_DIR):
    model_id = "kristaller486/dots.ocr-1.5" if USE_V15 else "rednote-hilab/dots.mocr"
    print(f"⬇️  Downloading {model_id} (~4GB, 5–10 phút)...")
    !python3 tools/download_model.py --name {model_id}
    print("✅ Model downloaded")
else:
    print("✅ Model đã có sẵn")

## Cell 3 — Patch model code cho T4 (chạy 1 lần/session)

In [ ]:
import os

def patch_file(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    new_lines = []
    modified = False
    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.lstrip()
        indent = line[:len(line) - len(stripped)]
        if any(f'Auto{t}.register' in line for t in ['Config','Model','Processor','Tokenizer']):
            if i == 0 or 'try:' not in lines[i-1]:
                new_lines += [f'{indent}try:\n', f'{indent}    {stripped}',
                              f'{indent}except (ValueError, AssertionError):\n',
                              f'{indent}    pass  # already registered\n']
                modified = True
                i += 1
                continue
        new_lines.append(line)
        i += 1
    if modified:
        with open(file_path, 'w') as f:
            f.writelines(new_lines)
    return modified

patched = []
for root, _, files in os.walk('./weights/DotsMOCR'):
    for fname in files:
        if fname.endswith('.py'):
            fpath = os.path.join(root, fname)
            if patch_file(fpath):
                patched.append(fname)
                print(f'✅ Patched: {fname}')

print(f'Done — patched {len(patched)} file(s)')

# --- Patch flash_attn import (T4 không hỗ trợ flash-attn 2) ---
FLASH_FALLBACK = """
try:
    from flash_attn import flash_attn_varlen_func
except ImportError:
    import torch, math
    def flash_attn_varlen_func(q, k, v, cu_seqlens_q, cu_seqlens_k,
                               max_seqlen_q, max_seqlen_k,
                               dropout_p=0.0, softmax_scale=None,
                               causal=False, **kwargs):
        if softmax_scale is None:
            softmax_scale = 1.0 / math.sqrt(q.shape[-1])
        outputs = []
        for b in range(len(cu_seqlens_q) - 1):
            qs, qe = cu_seqlens_q[b].item(), cu_seqlens_q[b+1].item()
            ks, ke = cu_seqlens_k[b].item(), cu_seqlens_k[b+1].item()
            qb = q[qs:qe].transpose(0,1).unsqueeze(0)
            kb = k[ks:ke].transpose(0,1).unsqueeze(0)
            vb = v[ks:ke].transpose(0,1).unsqueeze(0)
            out = torch.nn.functional.scaled_dot_product_attention(
                qb, kb, vb, scale=softmax_scale, is_causal=causal)
            outputs.append(out.squeeze(0).transpose(0,1))
        return torch.cat(outputs, dim=0)
"""

for root, _, files in os.walk('./weights/DotsMOCR'):
    for fname in files:
        if fname.endswith('.py'):
            fpath = os.path.join(root, fname)
            with open(fpath, 'r') as f:
                content = f.read()
            if 'from flash_attn import flash_attn_varlen_func' in content and 'except ImportError' not in content:
                content = content.replace(
                    'from flash_attn import flash_attn_varlen_func',
                    FLASH_FALLBACK.strip()
                )
                with open(fpath, 'w') as f:
                    f.write(content)
                print(f'✅ Patched flash_attn: {fname}')

import shutil
cache_dir = os.path.expanduser('~/.cache/huggingface/modules/transformers_modules')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print('✅ HuggingFace modules cache cleared')

## Cell 3b — Fix libcuda + FlashInfer (chạy 1 lần/session)

In [ ]:
import subprocess, os

# --- 1. libcuda.so stub (compile-time only, mỗi session) ---
with open('/tmp/cuda_stub.c', 'w') as f:
    f.write('void _cuda_stub_(void) {}\n')
r = subprocess.run(
    ['gcc', '-shared', '-fPIC', '-o', '/tmp/libcuda.so', '/tmp/cuda_stub.c'],
    capture_output=True, text=True
)
if r.returncode == 0:
    subprocess.run(['rm', '-rf', '/root/.cache/flashinfer'], capture_output=True)
    os.environ['LIBRARY_PATH'] = '/tmp'
    print('✅ libcuda.so stub created, cache cleared')
else:
    print(f'❌ gcc failed: {r.stderr}')

# --- 2. Patch vLLM FlashInfer: SDPA fallback for T4 sm_75 ---
fi_path = '/usr/local/lib/python3.12/dist-packages/vllm/v1/attention/backends/flashinfer.py'
_MARKER = '# _SDPA_FALLBACK_PATCHED_'
try:
    with open(fi_path) as f:
        content = f.read()
    if _MARKER in content:
        print('✅ flashinfer.py already patched')
    else:
        old_block = (
            '                    prefill_wrapper.run(\n'
            '                        prefill_query,\n'
            '                        kv_cache_permute,\n'
            '                        k_scale=layer._k_scale_float,\n'
            '                        v_scale=layer._v_scale_float,\n'
            '                        out=out_prefill,\n'
            '                        kv_cache_sf=kv_cache_sf,\n'
            '                    )'
        )
        new_block = (
            '                    ' + _MARKER + '\n'
            '                    try:\n'
            '                        prefill_wrapper.run(\n'
            '                            prefill_query,\n'
            '                            kv_cache_permute,\n'
            '                            k_scale=layer._k_scale_float,\n'
            '                            v_scale=layer._v_scale_float,\n'
            '                            out=out_prefill,\n'
            '                            kv_cache_sf=kv_cache_sf,\n'
            '                        )\n'
            '                    except Exception:\n'
            '                        import torch.nn.functional as _F\n'
            '                        _q = prefill_query.float()\n'
            '                        _k = key[num_decode_tokens:].float()\n'
            '                        _v = value[num_decode_tokens:].float()\n'
            '                        _q_t = _q.transpose(0, 1).unsqueeze(0)\n'
            '                        _k_t = _k.transpose(0, 1).unsqueeze(0)\n'
            '                        _v_t = _v.transpose(0, 1).unsqueeze(0)\n'
            '                        _nh, _nkvh = _q_t.shape[1], _k_t.shape[1]\n'
            '                        if _nh != _nkvh:\n'
            '                            _k_t = _k_t.repeat_interleave(_nh // _nkvh, dim=1)\n'
            '                            _v_t = _v_t.repeat_interleave(_nh // _nkvh, dim=1)\n'
            '                        _out = _F.scaled_dot_product_attention(\n'
            '                            _q_t, _k_t, _v_t,\n'
            '                            is_causal=True, scale=self.scale)\n'
            '                        out_prefill.copy_(\n'
            '                            _out.squeeze(0).transpose(0, 1).to(out_prefill.dtype))'
        )
        if old_block in content:
            content = content.replace(old_block, new_block)
            with open(fi_path, 'w') as f:
                f.write(content)
            print('✅ flashinfer.py patched — SDPA fallback added (T4/sm_75)')
        else:
            print('⚠️  Block không tìm thấy — vLLM version mới có thể không cần patch này')
except FileNotFoundError:
    print(f'⚠️  Không tìm thấy {fi_path} — bỏ qua')

## Cell 4a — Tạo ngrok tunnel

> Chạy **1 lần/session**. KHÔNG chạy lại khi vLLM crash.

In [ ]:
import os
from pyngrok import ngrok, conf

# Lấy token tại: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ""  # <-- dán token vào đây

if not NGROK_TOKEN:
    print('⚠️  Chưa có token! Dán vào NGROK_TOKEN = "..." rồi chạy lại')
else:
    conf.get_default().auth_token = NGROK_TOKEN

    ngrok.kill()
    import time; time.sleep(2)
    tunnel = ngrok.connect(8000, bind_tls=True)
    public_url = tunnel.public_url

    with open('/tmp/tunnel_url.txt', 'w') as f:
        f.write(public_url)

    print('\n' + '='*60)
    print(f'🌐 URL: {public_url}')
    print('='*60)
    print('👆 Dán URL này vào ô Server URL trong UI local!')
    print('\n⚠️  KHÔNG chạy lại Cell 4a khi vLLM crash!')
    print('   → Chỉ chạy lại Cell 4b — URL sẽ KHÔNG đổi.\n')

## Cell 4b — Khởi động vLLM server

> Chạy lại cell này khi vLLM crash — URL ngrok KHÔNG đổi.

In [ ]:
import subprocess, time, requests, os

try:
    with open('/tmp/tunnel_url.txt') as f:
        public_url = f.read().strip()
    print(f'🔗 Dùng URL: {public_url}')
except FileNotFoundError:
    print('⚠️  Chưa có URL — chạy Cell 4a trước!')
    public_url = None

if public_url:
    env = os.environ.copy()
    env['VLLM_USE_V1']              = '0'
    env['VLLM_ATTENTION_BACKEND']   = 'XFORMERS'
    env['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
    env['LIBRARY_PATH']             = '/tmp'

    subprocess.run(['pkill', '-f', 'vllm serve'], capture_output=True)
    time.sleep(3)

    log_file = open('/tmp/vllm.log', 'w')
    proc = subprocess.Popen([
        'vllm', 'serve', './weights/DotsMOCR',
        '--tensor-parallel-size',        '1',
        '--gpu-memory-utilization',      '0.95',
        '--max-model-len',               '32768',
        '--dtype',                       'half',
        '--chat-template-content-format','string',
        '--served-model-name',           'model',
        '--trust-remote-code',
        '--enforce-eager',
        '--port',                        '8000',
    ], stdout=log_file, stderr=log_file, env=env, preexec_fn=os.setsid)

    print('⏳ Đang khởi động vLLM (3–5 phút lần đầu)...')
    for i in range(120):
        time.sleep(5)
        if proc.poll() is not None:
            log_file.flush()
            print(f'❌ Server crash! Exit: {proc.poll()}')
            with open('/tmp/vllm.log') as f:
                lines = f.readlines()
            errs = [l for l in lines if any(k in l for k in ['ERROR','ValueError','RuntimeError','failed','OOM'])]
            print(''.join(errs[-20:]))
            break
        try:
            if requests.get('http://localhost:8000/v1/models', timeout=3).status_code == 200:
                print(f'\n✅ vLLM sẵn sàng sau {(i+1)*5}s!')
                print(f'🌐 URL: {public_url}')
                break
        except:
            if (i+1) % 6 == 0:
                print(f'   [{(i+1)*5}s] Đang khởi động...')
    else:
        print('⚠️  Timeout — xem log: open("/tmp/vllm.log").readlines()[-30:]')

## Cell 5 — Keep-alive

> Giữ cell này chạy suốt session để Kaggle không timeout.

In [ ]:
import time, requests, subprocess
from pyngrok import ngrok

print('🔄 Keep-alive đang chạy — giữ cell này running để Kaggle không timeout.')
print('   Nhấn Stop để dừng.\n')

count    = 0
failures = 0

while True:
    # In dấu · mỗi 10s → Kaggle idle-detector không kill cell
    for _tick in range(5):
        time.sleep(10)
        print('·', end='', flush=True)
    time.sleep(10)
    print()
    count += 1

    # Kiểm tra vLLM
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=5)
        if r.status_code == 200:
            failures = 0
            print(f'   [{count:3d} phút] ✅ vLLM OK')
        else:
            failures += 1
            print(f'   [{count:3d} phút] ⚠️  HTTP {r.status_code} (lần {failures})')
    except Exception as e:
        failures += 1
        print(f'   [{count:3d} phút] ❌ vLLM không phản hồi (lần {failures}): {e}')
        if failures >= 3:
            print('\n⛔ vLLM đã crash. → Chạy lại Cell 4b (URL KHÔNG đổi!)')
            break

    # Kiểm tra ngrok mỗi 5 phút (double-check trước khi reconnect)
    if count % 5 == 0:
        try:
            tunnels = ngrok.get_tunnels()
            if not tunnels:
                print(f'   [{count:3d} phút] ⚠️  Ngrok trống — chờ 30s...')
                time.sleep(30)
                tunnels = ngrok.get_tunnels()
                if not tunnels:
                    t = ngrok.connect(8000, bind_tls=True)
                    new_url = t.public_url
                    with open('/tmp/tunnel_url.txt', 'w') as f:
                        f.write(new_url)
                    print(f'\n🔄 [{count:3d} phút] Ngrok reconnect!')
                    print(f'   🌐 URL MỚI: {new_url}')
                    print('   👆 Cập nhật URL trong UI local!\n')
                else:
                    print(f'   [{count:3d} phút] ✅ Ngrok OK')
        except Exception as e:
            print(f'   [{count:3d} phút] ⚠️  Ngrok lỗi: {e}')